# Error-Gnomark Tutorial: Parts 1, 2, & 3

Welcome! This notebook combines the first three parts of the `errorgnomark` tutorial.

- **Part 1** introduces the main `QuantumCircuit` class.
- **Part 2** dives into the `Gate` class, the fundamental building block of circuits.
- **Part 3** explores the powerful `GateSet` classes for generating structured random circuits for experiments like Randomized Benchmarking (RB) and Cross-Entropy Benchmarking (XEB).

### **Important First Step**

Please run the setup cell below. It imports all the necessary classes for the entire tutorial.

In [1]:
# SETUP CELL: Run this first!

import numpy as np
from egm.core.circuits import QuantumCircuit, Gate

print("Setup complete. QuantumCircuit and Gate classes are imported and ready to use.")

Setup complete. QuantumCircuit and Gate classes are imported and ready to use.


---

# Part 1: The `QuantumCircuit` Class - Your Canvas

The `QuantumCircuit` class is the main object you will interact with. It acts as a container for your quantum gates and operations, organized in a sequence of "moments".

## 1.1 Creating a Quantum Circuit

Creating a circuit is simple. You just need to specify the number of qubits it will have.

In [2]:
# Create a quantum circuit with 3 qubits.
qubit_indices = [3]
qc = QuantumCircuit(qubit_indices)

print(f"Successfully created a QuantumCircuit with {qc.num_qubits} qubits.")

Successfully created a QuantumCircuit with 1 qubits.


## 1.2 Adding Gates to the Circuit

You can add gates using the `add_gate()` method. This method is flexible and allows you to specify the gate by its name and the qubits it acts on.

Let's build a simple Bell state preparation circuit: a Hadamard gate on qubit 0, followed by a CNOT gate with qubit 0 as control and qubit 1 as target.

In [3]:
# Assuming your file structure is correct, you can import like this
from egm.core.circuits.circuit import QuantumCircuit, Gate
import numpy as np # Import numpy for potential future use

# --- Correct Code ---

# 1. Create a QuantumCircuit object.
#    Key fix: List all the qubits that will be used here.
#    Since we are using CNOT(0, 1), we must include both 0 and 1.
print("Initializing QuantumCircuit with qubits [0, 1]...")
qc = QuantumCircuit(qubits=[0, 1])

# 2. Create a Gate object for the Hadamard gate and add it to the circuit.
h_gate = Gate('h', (0,))
qc.add_gate(h_gate)

# 3. Create a Gate object for the CNOT gate and add it to the circuit.
cnot_gate = Gate('cnot', (0, 1))
qc.add_gate(cnot_gate)

print("Successfully added H and CNOT gates to the circuit.")

Initializing QuantumCircuit with qubits [0, 1]...
Successfully added H and CNOT gates to the circuit.


## 1.3 Viewing the Circuit

There are two primary ways to see the circuit you've built:

1.  `print(qc)`: This gives you a simple, text-based summary of the gates in each moment.
2.  `qc.draw()`: This provides a more traditional, ASCII-art circuit diagram.

In [4]:
from egm.core.circuits import visualization

# 4. Draw the ASCII diagram of the circuit.
#    This will now run correctly because the drawing function knows that qubit 1 exists.
print("\n--- ASCII Diagram ---")
qc.draw()


--- ASCII Diagram ---
q0:     ┤  H  ├───●───
q1:     ──────────⊕───


---

# Part 2: The `Gate` Class - The Building Blocks

Every quantum circuit is composed of individual operations, or "gates". In `errorgnomark.circuits`, each of these operations is represented by an instance of the `Gate` class.

A key design principle of the `Gate` class is **immutability**. Once a `Gate` object is created, it cannot be changed. This makes the code safer and more predictable.

## 2.1 Creating Gates

To create a `Gate`, you need to provide its `name` and the `qubits` it acts on. You can also provide `params` for parameterized gates or set the `is_measurement` flag.

In [5]:
# A single-qubit gate: Hadamard on qubit 0
h_gate = Gate(name='h', qubits=(0,))
print(f"Single-qubit gate: {h_gate}")

# A two-qubit gate: CNOT with control qubit 1 and target qubit 2
cnot_gate = Gate(name='cnot', qubits=(1, 2))
print(f"Multi-qubit gate:  {cnot_gate}")

# A parameterized gate: RX gate on qubit 0 with an angle of π/2
rx_gate = Gate(name='rx', qubits=(0,), params=(np.pi / 2,))
print(f"Parameterized gate: {rx_gate}")

# A measurement operation on qubit 3
measure_op = Gate(name='measure', qubits=(3,), is_measurement=True)
print(f"Measurement op:    {measure_op}")

Single-qubit gate: Gate(name='h', qubits=(0,))
Multi-qubit gate:  Gate(name='cnot', qubits=(1, 2))
Parameterized gate: Gate(name='rx', qubits=(0,), params=(1.5707963267948966,))
Measurement op:    Gate(name='measure', qubits=(3,))


## 2.2 Gate Attributes

Each `Gate` object has several useful attributes:

- `name` (str): The name of the gate.
- `qubits` (Tuple[int, ...]): The qubit indices the gate acts upon.
- `params` (Tuple[Any, ...]): A tuple of parameters.
- `is_measurement` (bool): A flag for measurement operations.
- `arity` (property): The number of qubits the gate acts on.

In [6]:
# Let's inspect the rx_gate we created earlier

print(f"Name: {rx_gate.name}")
print(f"Qubits: {rx_gate.qubits}")
print(f"Parameters: {rx_gate.params}")
print(f"Arity (number of qubits): {rx_gate.arity}")
print(f"Is it a measurement?: {rx_gate.is_measurement}")

Name: rx
Qubits: (0,)
Parameters: (1.5707963267948966,)
Arity (number of qubits): 1
Is it a measurement?: False


## 2.3 Immutability and Hashability

Because `Gate` objects are immutable, they are also **hashable**. This means you can use them as keys in a dictionary or add them to a set. This is extremely useful for circuit analysis, such as counting unique gates.

In [7]:
# Create several gate instances, some of which are identical
gate1 = Gate('h', (0,))
gate2 = Gate('cnot', (0, 1))
gate3 = Gate('h', (0,)) # Identical to gate1
gate4 = Gate('h', (1,)) # Different from gate1 (acts on a different qubit)

# Let's use a set to find the unique gates
unique_gates = {gate1, gate2, gate3, gate4}

print(f"Original number of gates: 4")
print(f"Number of unique gates: {len(unique_gates)}")
print("Set of unique gates:")
for gate in unique_gates:
    print(f"  {gate}")

# We can also use them as dictionary keys to count occurrences
gate_counts = {}
all_gates_list = [gate1, gate2, gate3, gate4]
for g in all_gates_list:
    gate_counts[g] = gate_counts.get(g, 0) + 1

print("\nGate counts:")
for gate, count in gate_counts.items():
    print(f"  {gate}: {count}")

Original number of gates: 4
Number of unique gates: 3
Set of unique gates:
  Gate(name='h', qubits=(1,))
  Gate(name='h', qubits=(0,))
  Gate(name='cnot', qubits=(0, 1))

Gate counts:
  Gate(name='h', qubits=(0,)): 2
  Gate(name='cnot', qubits=(0, 1)): 1
  Gate(name='h', qubits=(1,)): 1


## 2.4 Gate Inversion

The `Gate` class has a convenient `.inverse()` method. It returns a **new** `Gate` object representing the inverse operation, without modifying the original.

In [8]:
# 1. Self-inverse gates (e.g., H, X, CNOT)
h = Gate('h', (0,))
h_inv = h.inverse()
print(f"Original: {h}, Inverse: {h_inv}, Are they equal? {h == h_inv}")

# 2. Gates with specific inverses (e.g., S and Sdg)
s = Gate('s', (1,))
s_inv = s.inverse()
print(f"Original: {s}, Inverse: {s_inv}")

# 3. Parameterized gates (e.g., RX(θ) -> RX(-θ))
rx = Gate('rx', (2,), params=(np.pi/4,))
rx_inv = rx.inverse()
print(f"Original: {rx}, Inverse: {rx_inv}")

# 4. Generic gates (appends 'dg' for dagger)
my_gate = Gate('my_custom_gate', (3,))
my_gate_inv = my_gate.inverse()
print(f"Original: {my_gate}, Inverse: {my_gate_inv}")

Original: Gate(name='h', qubits=(0,)), Inverse: Gate(name='h', qubits=(0,)), Are they equal? True
Original: Gate(name='s', qubits=(1,)), Inverse: Gate(name='sdg', qubits=(1,))
Original: Gate(name='rx', qubits=(2,), params=(0.7853981633974483,)), Inverse: Gate(name='rx', qubits=(2,), params=(-0.7853981633974483,))
Original: Gate(name='my_custom_gate', qubits=(3,)), Inverse: Gate(name='my_custom_gatedg', qubits=(3,))


---

# Part 3: Generating Gate Layers with `GateSet`s

Often in quantum computing, especially when characterizing hardware, we don't build one specific circuit. Instead, we need to generate many *random* circuits that follow a certain structure. The `GateSet` classes are designed for this purpose.

A `GateSet` is a blueprint for generating layers of gates, commonly used in experiments like:
- **Randomized Benchmarking (RB)**: To measure the average error of a set of gates.
- **Cross-Entropy Benchmarking (XEB)**: To demonstrate quantum advantage by comparing output distributions to a classical simulation.

### **Part 3 Setup**

First, run the cell below. It defines all the `GateSet` classes and the `get_gate_set` factory function that we will use in this section. This makes our notebook self-contained.

## 3.1 Using the `get_gate_set` Factory

The easiest way to get started is with the `get_gate_set` factory function. You simply pass the name of the desired gate set, and it returns an instance of the correct class.

In [9]:
# Keep the original import statement as is
from egm.core.circuits import gate_sets

# --- Modification here ---
# Call the get_gate_set function via the gate_sets module object
clifford_set = gate_sets.get_gate_set('clifford')
print(f"Got gate set: {clifford_set}")

# Similarly, modify here
xeb_set = gate_sets.get_gate_set('universal_xeb')
print(f"Got gate set: {xeb_set}")

Got gate set: <errorgnomark.circuits.gate_sets.CliffordGateSet object at 0x118743a00>
Got gate set: <errorgnomark.circuits.gate_sets.UniversalXEBGateSet object at 0x11cbaf310>


As you can see, the `GateSet` classes provide a high-level, convenient API for constructing complex, structured random circuits, which is a common requirement for quantum hardware characterization.

---

## Tutorial Complete!

You have now learned about the three most fundamental components in `errorgnomark.circuits`:

- **`QuantumCircuit`**: The container for your algorithm.
- **`Gate`**: The immutable, hashable building blocks of the circuit.
- **`GateSet`**: Blueprints for generating structured random gate layers for experiments.

With this knowledge, you are ready to build, analyze, and generate a wide variety of quantum circuits.